In [45]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.model_selection import train_test_split
import os
import warnings
warnings.filterwarnings("ignore")

In [46]:
RAW_DIR = r"D:\projects\Healthcare\data\raw\\"
SAMPLE_DIR = r"D:\projects\Healthcare\data\raw_sample"

In [47]:
datasets = {
    "diabetes" : {"file":   "diabetes_data.csv",  "target": "diabetes", "size": 20000},
    "cardio"   : {"file":    "cardio_data.csv",       "target": "cardio",   "size": 20000},
    "stroke"   : {"file":    "stroke_data.csv",           "target": "Diagnosis", "size":15000},
}

print("Setup ready. Datasets:", list(datasets.keys()))

Setup ready. Datasets: ['diabetes', 'cardio', 'stroke']


In [48]:

def slice_dataset(name, info):
    """Stratified-ah slice panni return (ratio preserve)."""
    path = os.path.join(RAW_DIR, info["file"])
    df = pd.read_csv(path)
    target, size = info["target"], info["size"]

    print(f"\n{'='*55}\n  {name.upper()}\n{'='*55}")
    print(f"Original : {df.shape[0]} rows x {df.shape[1]} cols")
    orig = (df[target].value_counts(normalize=True) * 100).round(2)

    if df.shape[0] <= size:
        print(f"--> Already {df.shape[0]} rows (<= {size}). As-is.")
        sliced = df
    else:
        sliced, _ = train_test_split(df, train_size=size,
                                     stratify=df[target], random_state=42)
        print(f"--> Sliced to {size} rows (stratified).")

    new = (sliced[target].value_counts(normalize=True) * 100).round(2)
    print(f"Ratio ('{target}')  original -> sliced:")
    for cls in orig.index:
        print(f"    {str(cls):<14}: {orig[cls]:>6}% -> {new.get(cls, 0):>6}%")
    return sliced


In [49]:
os.makedirs(SAMPLE_DIR, exist_ok=True)  
sliced_datasets = {}
for name, info in datasets.items():
    sliced = slice_dataset(name, info)
    out_path = os.path.join(SAMPLE_DIR, f"{name}_sliced.csv")
    sliced.to_csv(out_path, index=False)
    sliced_datasets[name] = sliced
    print(f"Saved --> {out_path}")


  DIABETES
Original : 100000 rows x 9 cols
--> Sliced to 20000 rows (stratified).
Ratio ('diabetes')  original -> sliced:
    0             :   91.5% ->   91.5%
    1             :    8.5% ->    8.5%
Saved --> D:\projects\Healthcare\data\raw_sample\diabetes_sliced.csv

  CARDIO
Original : 68205 rows x 17 cols
--> Sliced to 20000 rows (stratified).
Ratio ('cardio')  original -> sliced:
    0             :  50.63% ->  50.63%
    1             :  49.37% ->  49.37%
Saved --> D:\projects\Healthcare\data\raw_sample\cardio_sliced.csv

  STROKE
Original : 15000 rows x 22 cols
--> Already 15000 rows (<= 15000). As-is.
Ratio ('Diagnosis')  original -> sliced:
    No Stroke     :  50.21% ->  50.21%
    Stroke        :  49.79% ->  49.79%
Saved --> D:\projects\Healthcare\data\raw_sample\stroke_sliced.csv


In [52]:
SAMPLE_DIR = r"D:\projects\Healthcare\data\raw_sample"
test_file = os.path.join(SAMPLE_DIR, "diabetes_sliced.csv")

df = pd.read_csv(test_file)

print("Shape(rows, cols)", df.shape)
print("\nColumns:", df.columns.tolist())

print(df.head())

Shape(rows, cols) (20000, 9)

Columns: ['gender', 'age', 'hypertension', 'heart_disease', 'smoking_history', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'diabetes']
   gender   age  hypertension  heart_disease smoking_history    bmi  \
0  Female  36.0             0              0         No Info  27.32   
1  Female  56.0             0              0          former  23.43   
2  Female  35.0             0              0            ever  22.67   
3  Female  15.0             0              0         No Info  27.65   
4  Female  29.0             0              0           never  22.39   

   HbA1c_level  blood_glucose_level  diabetes  
0          4.0                  159         0  
1          4.0                  160         0  
2          4.5                   85         0  
3          4.5                   80         0  
4          5.0                   85         0  


Logic Validation

In [56]:
def validate_dataset(df):
    """Check whether dataset is correct, If problems is there return as list"""
    issues = []

    if df.shape[0] == 0:
        issues.append("Dataset has 0 rows")

    if df.shape[1] == 0:
        issues.append("Dataset has 0 columns")

    if df.shape[1] < 2:
        issues.append("Dataset has less than 2 columns")

    empty_cols = [c for c in df.columns if df[c].isna().all()]
    if empty_cols:
        issues.append(f"Full empty columns:{empty_cols}")

    is_valid = len(issues) == 0
    return is_valid, issues

is_valid, issues = validate_dataset(df)
print("Is it valid:", is_valid)
print("Issues:", issues if issues else "No issues present")



Is it valid: True
Issues: No issues present


In [69]:
def analyze_dataset(df):
    """Generate a complete summary of the dataset."""
    try:
        analysis = {
            "n_rows": df.shape[0],
            "n_cols": df.shape[1],
            "duplicate_rows": int(df.duplicated().sum()),
            "memory_kb": round(df.memory_usage(deep=True).sum() / 1024, 2),
            "columns": [],
        }

        for col in df.columns:
            col_info = {
                "name": col,
                "dtype": str(df[col].dtype),
                "missing": int(df[col].isna().sum()),
                "missing_pct": round(df[col].isna().mean() * 100, 2),
                "unique": int(df[col].nunique()),
            }
            analysis["columns"].append(col_info)

        return analysis

    except Exception as e:
        print(f"Error during analysis: {e}")
        return None


# Test
result = analyze_dataset(df)

print(f"Rows: {result['n_rows']}, Cols: {result['n_cols']}")
print(f"Duplicate rows: {result['duplicate_rows']}")
print(f"Memory: {result['memory_kb']} KB\n")

print(f"{'Column':<22}{'Type':<12}{'Missing':<10}{'Miss%':<8}{'Unique'}")
print("-" * 60)
for c in result["columns"]:
    print(f"{c['name']:<22}{c['dtype']:<12}{c['missing']:<10}{c['missing_pct']:<8}{c['unique']}")

Rows: 20000, Cols: 9
Duplicate rows: 196
Memory: 1631.38 KB

Column                Type        Missing   Miss%   Unique
------------------------------------------------------------
gender                str         0         0.0     3
age                   float64     0         0.0     102
hypertension          int64       0         0.0     2
heart_disease         int64       0         0.0     2
smoking_history       str         0         0.0     6
bmi                   float64     0         0.0     3156
HbA1c_level           float64     0         0.0     18
blood_glucose_level   int64       0         0.0     18
diabetes              int64       0         0.0     2
